![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 01 | Exploratory Data Analysis

*Project 1 — SQL: From Data to Insight*

**Team:**
**Dataset:**

---

Understanding your data before you design anything. You cannot decide which tables you need until you know which columns repeat, which are broken, and which identify a row.

> **Done when:** you can name the 3+ tables you are going to build, you have written down at least 2 research questions, and you know every data-quality problem notebook 02 will have to fix.

The sections below are a suggested route, not a form to fill in. Add sections, drop them, reorder them — what is graded is the analysis, not whether you kept the scaffold.

---
## 0. Setup

In [1]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.functions import *

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

---
## 1. Load the raw data

Run `python download_data.py <payments|airbnb|retail>` from the repo root first. Your files land in `data/raw/`, which is gitignored.

In [6]:

demo = pd.read_sas("../data/raw/DEMO_L.XPT", format="xport")
activity = pd.read_sas("../data/raw/PAQ_L.XPT", format="xport")
body = pd.read_sas("../data/raw/BMX_L.XPT", format="xport")
bp = pd.read_sas("../data/raw/BPXO_L.XPT", format="xport")

---
## 2. First look

Shape, column types, a handful of rows. What is one row of this data?

In [9]:
#First renaming the column to undestand the data better
demo = demo.rename(columns={
    "SEQN": "participant_id",
    "SDDSRVYR": "survey_cycle",
    "RIDSTATR": "interview_exam_status",
    "RIAGENDR": "sex",
    "RIDAGEYR": "age_years",
    "RIDAGEMN": "age_months_screening",
    "RIDRETH1": "race_ethnicity",
    "RIDRETH3": "race_ethnicity_asian_detail",
    "RIDEXMON": "exam_period",
    "RIDEXAGM": "age_months_exam",
    "DMQMILIZ": "military_service",
    "DMDBORN4": "country_of_birth",
    "DMDYRUSR": "years_in_us",
    "DMDEDUC2": "education_level",
    "DMDMARTZ": "marital_status",
    "RIDEXPRG": "pregnancy_status",
    "DMDHHSIZ": "household_size",
    "DMDHRGND": "household_reference_sex",
    "DMDHRAGZ": "household_reference_age",
    "DMDHREDZ": "household_reference_education",
    "DMDHRMAZ": "household_reference_marital_status",
    "DMDHSEDZ": "household_spouse_education",
    "WTINT2YR": "interview_weight",
    "WTMEC2YR": "exam_weight",
    "SDMVSTRA": "survey_stratum",
    "SDMVPSU": "survey_psu",
    "INDFMPIR": "income_poverty_ratio"
})
demo.head()

,participant_id,survey_cycle,interview_exam_status,sex,age_years,age_months_screening,race_ethnicity,race_ethnicity_asian_detail,exam_period,age_months_exam,military_service,country_of_birth,years_in_us,education_level,marital_status,pregnancy_status,household_size,household_reference_sex,household_reference_age,household_reference_education,household_reference_marital_status,household_spouse_education,interview_weight,exam_weight,survey_stratum,survey_psu,income_poverty_ratio
0,130378.0,12.0,2.0,1.0,43.0,NaN,5.0,6.0,2.0,NaN,2.0,2.0,6.0,5.0,1.0,NaN,4.0,NaN,NaN,NaN,NaN,NaN,50055.450807,54374.463898,173.0,2.0,5.00
1,130379.0,12.0,2.0,1.0,66.0,NaN,3.0,3.0,2.0,NaN,2.0,1.0,NaN,5.0,1.0,NaN,2.0,NaN,NaN,NaN,NaN,NaN,29087.450605,34084.721548,173.0,2.0,5.00
2,130380.0,12.0,2.0,2.0,44.0,NaN,2.0,2.0,1.0,NaN,2.0,2.0,6.0,3.0,1.0,2.0,7.0,NaN,NaN,NaN,NaN,NaN,80062.674301,81196.277992,174.0,1.0,1.41
3,130381.0,12.0,2.0,2.0,5.0,NaN,5.0,7.0,1.0,71.0,NaN,1.0,NaN,NaN,NaN,NaN,2.0,2.0,2.0,2.0,3.0,NaN,38807.268902,55698.607106,182.0,2.0,1.53
4,130382.0,12.0,2.0,1.0,2.0,NaN,3.0,3.0,2.0,34.0,NaN,1.0,NaN,NaN,NaN,NaN,4.0,2.0,2.0,3.0,1.0,2.0,30607.519774,36434.146346,182.0,2.0,3.60


In [ ]:
#Extracting the columns that I think is important to respond the scientific questions.
demo = demo[
    [
        "participant_id",
        "age_years",
        "sex",
        "exam_weight",
        "education_level",
        "income_poverty_ratio"
    ]
]
demo

,participant_id,age_years,sex,exam_weight,education_level,income_poverty_ratio
0,130378.0,43.0,1.0,54374.463898,5.0,5.00
1,130379.0,66.0,1.0,34084.721548,5.0,5.00
2,130380.0,44.0,2.0,81196.277992,3.0,1.41
3,130381.0,5.0,2.0,55698.607106,NaN,1.53
4,130382.0,2.0,1.0,36434.146346,NaN,3.60
...,...,...,...,...,...,...
11928,142306.0,9.0,1.0,13459.129019,NaN,2.01
11929,142307.0,49.0,2.0,64962.328962,5.0,NaN
11930,142308.0,50.0,1.0,44367.534132,4.0,1.95
11931,142309.0,40.0,1.0,46249.361849,4.0,3.11


In [14]:
#renaming the activity to a more understandeble column names
activity = activity.rename(columns={
    "SEQN": "participant_id",
    "PAD790Q": "moderate_activity_frequency",
    "PAD790U": "moderate_activity_unit",
    "PAD800": "moderate_activity_minutes",
    "PAD810Q": "vigorous_activity_frequency",
    "PAD810U": "vigorous_activity_unit",
    "PAD820": "vigorous_activity_minutes",
    "PAD680": "sitting_minutes_per_day"
})
activity.head()

,participant_id,moderate_activity_frequency,moderate_activity_unit,moderate_activity_minutes,vigorous_activity_frequency,vigorous_activity_unit,vigorous_activity_minutes,sitting_minutes_per_day
0,130378.0,3.000000e+00,b'W',45.0,3.000000e+00,b'W',45.0,360.0
1,130379.0,4.000000e+00,b'W',45.0,3.000000e+00,b'W',45.0,480.0
2,130380.0,1.000000e+00,b'W',20.0,5.397605e-79,b'',NaN,240.0
3,130384.0,5.397605e-79,b'',NaN,5.397605e-79,b'',NaN,60.0
4,130385.0,1.000000e+00,b'D',90.0,1.000000e+00,b'W',60.0,180.0


In [ ]:
#Rename the Body columns
body = body.rename(columns={
    "SEQN": "participant_id",
    "BMDSTATS": "body_measure_status",

    "BMXWT": "weight_kg",
    "BMIWT": "weight_comment",

    "BMXRECUM": "recumbent_length_cm",
    "BMIRECUM": "recumbent_length_comment",

    "BMXHEAD": "head_circumference_cm",
    "BMIHEAD": "head_circumference_comment",

    "BMXHT": "height_cm",
    "BMIHT": "height_comment",

    "BMXBMI": "bmi",
    "BMDBMIC": "bmi_category_child_youth",

    "BMXLEG": "upper_leg_length_cm",
    "BMILEG": "upper_leg_length_comment",

    "BMXARML": "upper_arm_length_cm",
    "BMIARML": "upper_arm_length_comment",

    "BMXARMC": "arm_circumference_cm",
    "BMIARMC": "arm_circumference_comment",

    "BMXWAIST": "waist_circumference_cm",
    "BMIWAIST": "waist_circumference_comment",

    "BMXHIP": "hip_circumference_cm",
    "BMIHIP": "hip_circumference_comment"
})
body.head()

,participant_id,body_measure_status,weight_kg,weight_comment,recumbent_length_cm,recumbent_length_comment,head_circumference_cm,head_circumference_comment,height_cm,height_comment,bmi,bmi_category_child_youth,upper_leg_length_cm,upper_leg_length_comment,upper_arm_length_cm,upper_arm_length_comment,arm_circumference_cm,arm_circumference_comment,waist_circumference_cm,waist_circumference_comment,hip_circumference_cm,hip_circumference_comment
0,130378.0,1.0,86.9,NaN,NaN,NaN,NaN,NaN,179.5,NaN,27.0,NaN,42.8,NaN,42.0,NaN,35.7,NaN,98.3,NaN,102.9,NaN
1,130379.0,1.0,101.8,NaN,NaN,NaN,NaN,NaN,174.2,NaN,33.5,NaN,38.5,NaN,38.7,NaN,33.7,NaN,114.7,NaN,112.4,NaN
2,130380.0,1.0,69.4,NaN,NaN,NaN,NaN,NaN,152.9,NaN,29.7,NaN,38.5,NaN,35.5,NaN,36.3,NaN,93.5,NaN,98.0,NaN
3,130381.0,1.0,34.3,NaN,NaN,NaN,NaN,NaN,120.1,NaN,23.8,4.0,NaN,NaN,25.4,NaN,23.4,NaN,70.4,NaN,NaN,NaN
4,130382.0,3.0,13.6,NaN,NaN,1.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,1.0,NaN,1.0,NaN,NaN


In [ ]:
#The columns I need for the analysis
body = body[
    [
        "participant_id",
        "weight_kg",
        "height_cm",
        "bmi",
        "waist_circumference_cm",
        "hip_circumference_cm"
    ]
]
body

,participant_id,weight_kg,height_cm,bmi,waist_circumference_cm,hip_circumference_cm
0,130378.0,86.9,179.5,27.0,98.3,102.9
1,130379.0,101.8,174.2,33.5,114.7,112.4
2,130380.0,69.4,152.9,29.7,93.5,98.0
3,130381.0,34.3,120.1,23.8,70.4,NaN
4,130382.0,13.6,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
8855,142306.0,25.3,128.0,15.4,57.7,NaN
8856,142307.0,NaN,143.8,NaN,NaN,NaN
8857,142308.0,79.3,173.3,26.4,98.4,97.7
8858,142309.0,81.9,179.1,25.5,96.0,103.3


In [19]:
#Rename the blood pressure columns
bp = bp.rename(columns={
    "SEQN": "participant_id",

    "BPAOARM": "measurement_arm",
    "BPAOCSZ": "cuff_size_code",

    "BPXOSY1": "systolic_bp_1",
    "BPXODI1": "diastolic_bp_1",

    "BPXOSY2": "systolic_bp_2",
    "BPXODI2": "diastolic_bp_2",

    "BPXOSY3": "systolic_bp_3",
    "BPXODI3": "diastolic_bp_3",

    "BPXOPLS1": "pulse_1",
    "BPXOPLS2": "pulse_2",
    "BPXOPLS3": "pulse_3"
})
bp.head()

,participant_id,measurement_arm,cuff_size_code,systolic_bp_1,diastolic_bp_1,systolic_bp_2,diastolic_bp_2,systolic_bp_3,diastolic_bp_3,pulse_1,pulse_2,pulse_3
0,130378.0,b'R',4.0,135.0,98.0,131.0,96.0,132.0,94.0,82.0,79.0,82.0
1,130379.0,b'R',4.0,121.0,84.0,117.0,76.0,113.0,76.0,72.0,71.0,73.0
2,130380.0,b'R',4.0,111.0,79.0,112.0,80.0,104.0,76.0,84.0,83.0,77.0
3,130386.0,b'R',4.0,110.0,72.0,120.0,74.0,115.0,75.0,59.0,64.0,64.0
4,130387.0,b'R',4.0,143.0,76.0,136.0,74.0,145.0,78.0,80.0,80.0,77.0


## Blood Pressure Variables

The blood pressure dataset contains repeated measurements for each participant.

- **`measurement_arm`**: Indicates which arm was used for the blood pressure measurement.  
  - `L` = Left arm  
  - `R` = Right arm  
  This variable describes the measurement procedure and is **not required for the main analysis**.

- **`cuff_size_code`**: Indicates the size of the blood pressure cuff used. The cuff size is selected according to the participant's upper-arm circumference. This variable is also part of the measurement procedure and is **not required for the main analysis**.

- **`systolic_bp_1`, `systolic_bp_2`, `systolic_bp_3`**: Three systolic blood pressure measurements, reported in **mmHg**. Systolic blood pressure represents the pressure in the arteries when the heart contracts and pumps blood. It is the upper number in a blood pressure reading, for example **120** in 120/80 mmHg.

- **`diastolic_bp_1`, `diastolic_bp_2`, `diastolic_bp_3`**: Three diastolic blood pressure measurements, reported in **mmHg**. Diastolic blood pressure represents the pressure in the arteries when the heart relaxes between beats. It is the lower number in a blood pressure reading, for example **80** in 120/80 mmHg.

- **`pulse_1`, `pulse_2`, `pulse_3`**: Three pulse-rate measurements. Pulse rate represents the number of heartbeats per minute. Pulse is not essential for the main research questions, but it may be kept for an optional additional analysis.

For the main analysis, the repeated systolic and diastolic measurements will later be combined by calculating the **average systolic blood pressure** and **average diastolic blood pressure** for each participant.

In [ ]:
#Keeping the columns that needed
bp = bp[
    [
        "participant_id",
        "systolic_bp_1",
        "diastolic_bp_1",
        "systolic_bp_2",
        "diastolic_bp_2",
        "systolic_bp_3",
        "diastolic_bp_3"
    ]
]
bp.head()

,participant_id,systolic_bp_1,diastolic_bp_1,systolic_bp_2,diastolic_bp_2,systolic_bp_3,diastolic_bp_3
0,130378.0,135.0,98.0,131.0,96.0,132.0,94.0
1,130379.0,121.0,84.0,117.0,76.0,113.0,76.0
2,130380.0,111.0,79.0,112.0,80.0,104.0,76.0
3,130386.0,110.0,72.0,120.0,74.0,115.0,75.0
4,130387.0,143.0,76.0,136.0,74.0,145.0,78.0
...,...,...,...,...,...,...,...
7796,142306.0,NaN,NaN,NaN,NaN,NaN,NaN
7797,142307.0,127.0,75.0,132.0,73.0,131.0,72.0
7798,142308.0,106.0,65.0,106.0,69.0,112.0,74.0
7799,142309.0,127.0,81.0,125.0,82.0,128.0,81.0


---
## 3. Data quality

Missing values, duplicates, numbers stored as text, dates stored as strings. Write down what you find — this is the to-do list for notebook 02.

In [50]:
datasets = {
    "demographics": demo,
    "physical_activity": activity,
    "body_measures": body,
    "blood_pressure": bp
}

missing_summary = pd.DataFrame([
    {
        "dataset": name,
        "total_data": len(df),
        "total_missing": df.isna().any(axis=1).sum(), #number of rows that include missing value
        "missing_percent": round(df.isna().any(axis=1).sum() / df.size * 100, 2)
    }
    for name, df in datasets.items()
])

missing_summary

,dataset,total_data,total_missing,missing_percent
0,demographics,11933,5445,7.60
1,physical_activity,8153,4567,7.00
2,body_measures,8860,2111,3.97
3,blood_pressure,7801,323,0.59


- First I have to remove all Missing values from the data.

In [34]:
datasets = {
    "demographics": demo,
    "physical_activity": activity,
    "body_measures": body,
    "blood_pressure": bp
}

for name, df in datasets.items():
    print(name)
    print(
        "Duplicated participant IDs:",
        df["participant_id"].duplicated().sum()
    )
    print()

demographics
Duplicated participant IDs: 0

physical_activity
Duplicated participant IDs: 0

body_measures
Duplicated participant IDs: 0

blood_pressure
Duplicated participant IDs: 0



There is no Duplicates in each data sets.

In [35]:
for name, df in datasets.items():
    print("\n", name)
    print(df.dtypes)


 demographics
participant_id          float64
age_years               float64
sex                     float64
exam_weight             float64
education_level         float64
income_poverty_ratio    float64
dtype: object

 physical_activity
participant_id                 float64
moderate_activity_frequency    float64
moderate_activity_unit          object
moderate_activity_minutes      float64
vigorous_activity_frequency    float64
vigorous_activity_unit          object
vigorous_activity_minutes      float64
sitting_minutes_per_day        float64
dtype: object

 body_measures
participant_id            float64
weight_kg                 float64
height_cm                 float64
bmi                       float64
waist_circumference_cm    float64
hip_circumference_cm      float64
dtype: object

 blood_pressure
participant_id    float64
systolic_bp_1     float64
diastolic_bp_1    float64
systolic_bp_2     float64
diastolic_bp_2    float64
systolic_bp_3     float64
diastolic_bp_3    float64


There is no number recorded as string. and the object ones should be return to string in the cleaning process.
- second I have to decode the object columns to string values for day, week, month, year.

In [38]:
demo["sex"].value_counts(dropna=False)

sex
2.0    6358
1.0    5575
Name: count, dtype: int64

It seems there is no problem here

In [40]:
activity["moderate_activity_unit"].value_counts(dropna=False)


moderate_activity_unit
b'W'    4851
b''     1763
b'D'     985
b'M'     472
b'Y'      82
Name: count, dtype: int64

In [41]:
activity["vigorous_activity_unit"].value_counts(dropna=False)

vigorous_activity_unit
b''     4467
b'W'    2590
b'M'     711
b'D'     235
b'Y'     150
Name: count, dtype: int64

this values b'' are also odd and means no data recorded.

In [45]:
activity.loc[
    activity["vigorous_activity_unit"] == b"",
    "vigorous_activity_minutes"
]

2      NaN
3      NaN
6      NaN
7      NaN
9      NaN
        ..
8148   NaN
8149   NaN
8150   NaN
8151   NaN
8152   NaN
Name: vigorous_activity_minutes, Length: 4467, dtype: float64

Because those b'' producing NAN values in the other columns, so I guess these are included in NaN values can be removed. I have to check this again when the NaNs are removed.

---
## 4. Distributions

The numeric columns you care about, and the outliers you will have to decide about.

In [47]:

for name, df in datasets.items():
    print(f"\n{name}")

    numeric_cols = df.select_dtypes(include="number").columns

    for col in numeric_cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        n_outliers = (
            (df[col] < lower_bound) |
            (df[col] > upper_bound)
        ).sum()

        print(col, ":", n_outliers)


demographics
participant_id : 0
age_years : 0
sex : 0
exam_weight : 377
education_level : 11
income_poverty_ratio : 0

physical_activity
participant_id : 0
moderate_activity_frequency : 138
moderate_activity_minutes : 1072
vigorous_activity_frequency : 242
vigorous_activity_minutes : 569
sitting_minutes_per_day : 185

body_measures
participant_id : 0
weight_kg : 137
height_cm : 770
bmi : 189
waist_circumference_cm : 39
hip_circumference_cm : 218

blood_pressure
participant_id : 0
systolic_bp_1 : 143
diastolic_bp_1 : 93
systolic_bp_2 : 185
diastolic_bp_2 : 125
systolic_bp_3 : 178
diastolic_bp_3 : 106


After removing Na s then I will chceck this values and remove them in the data cleaning.

In [ ]:
demo_numerical_cols = [
    "age_years",
    "exam_weight",
    "income_poverty_ratio"
]
activity_numerical_cols = [
    "moderate_activity_frequency",
    "moderate_activity_minutes",
    "vigorous_activity_frequency",
    "vigorous_activity_minutes",
    "sitting_minutes_per_day"
]
body_numerical_cols = [
    "weight_kg",
    "height_cm",
    "bmi",
    "waist_circumference_cm",
    "hip_circumference_cm"
]
bp_numerical_cols = [
    "systolic_bp_1",
    "diastolic_bp_1",
    "systolic_bp_2",
    "diastolic_bp_2",
    "systolic_bp_3",
    "diastolic_bp_3"
]

---
## 5. Categorical columns

**This is the section that feeds tomorrow.** A column with few distinct values repeated over many rows is a lookup table waiting to happen. Find them.

In [49]:
categorical_cols_demo = [
    "sex"
]
categorical_cols_activity = [
    "moderate_activity_unit",
    "vigorous_activity_unit"
]

---
## 6. Research questions

At least two, written here. Specific, answerable with this data, and with an answer you could be wrong about. For each one, say what you expect and why.

1. How does average blood pressure differ across age groups and sex?
2. How do average blood pressure and BMI differ across physical-activity groups?

---
## 7. Into notebook 02

The tables you are going to build, and the problems you have to fix first.